In [1]:
# load data from website

from langchain_community.document_loaders import WebBaseLoader
loader = WebBaseLoader("https://docs.smith.langchain.com/tutorials/Administrators/manage_spend")
docs = loader.load()
print(docs[0].page_content[:1000])


C:\Users\yashw\AppData\Local\Temp\ipykernel_21916\1604024590.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import WebBaseLoader
USER_AGENT environment variable not set, consider setting it to identify your requests.


LangSmith overview - Docs by LangChainDocumentation IndexFetch the complete documentation index at: /llms.txtUse this file to discover all available pages before exploring further.Skip to main contentDocs by LangChain home pageHomeSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationLangSmith overviewHomeLangSmithAgent lifecycleLangSmith overviewCopy pageCopy pageLangSmith is a framework-agnostic platform for building, debugging, and deploying AI agents and LLM applications. Trace requests, evaluate outputs, test prompts, and manage deployments all in one place, with your agent stack.
​Get started
Create an accountSign up at smith.langchain.com (no credit card required).
You can log in with Google, GitHub, or email.Create an API keyGo to your Settings page → API Keys → Create API Key.
Copy the key and save it securely.Choose your integrationLangSmith works with many frameworks and providers. Browse available integrations to connect your stack.
Once your account and API k

In [2]:
pip install langchain-text-splitters

Note: you may need to restart the kernel to use updated packages.


In [3]:
# split into chunks
from langchain_text_splitters import RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
documents=text_splitter.split_documents(docs)
print('total chunks:', len(documents))
print(documents[0].page_content)

total chunks: 4
LangSmith overview - Docs by LangChainDocumentation IndexFetch the complete documentation index at: /llms.txtUse this file to discover all available pages before exploring further.Skip to main contentDocs by LangChain home pageHomeSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationLangSmith overviewHomeLangSmithAgent lifecycleLangSmith overviewCopy pageCopy pageLangSmith is a framework-agnostic platform for building, debugging, and deploying AI agents and LLM applications. Trace requests, evaluate outputs, test prompts, and manage deployments all in one place, with your agent stack.
​Get started
Create an accountSign up at smith.langchain.com (no credit card required).
You can log in with Google, GitHub, or email.Create an API keyGo to your Settings page → API Keys → Create API Key.
Copy the key and save it securely.Choose your integrationLangSmith works with many frameworks and providers. Browse available integrations to connect your stack.


In [4]:
from langchain_ollama import OllamaEmbeddings
embeddings = OllamaEmbeddings(model="nomic-embed-text")


In [5]:
from langchain_community.vectorstores import FAISS
vectorstore = FAISS.from_documents(documents, embeddings)
print('faiss index created')

faiss index created


In [6]:
# Similarity search
query = "What spending controls are available?"
results = vectorstore.similarity_search(query,k=3)
for doc in results:
    print(doc.page_content)
    print("="*100)

​More ways to build
FleetDesign and deploy AI agents visually without writing code.Build an agentPrompt engineeringIterate on prompts with built-in versioning and collaboration to ship improvements faster.Test your promptsLangSmith CLIQuery and manage traces, datasets, experiments, and more from the terminal.Use the CLIStudioUse a visual interface to design, test, and refine applications end-to-end.Develop with Studio
​Infrastructure
Platform setupUse LangSmith in managed cloud, in a self-hosted environment, or hybrid to match your infrastructure and compliance needs.Choose how to set up LangSmithSecurity & complianceLangSmith meets the highest standards of data security and privacy with HIPAA, SOC 2 Type 2, and GDPR compliance. Meet with our team to learn more or visit our Trust Center.Visit Trust Center
​Workflow
LangSmith combines observability, evaluation, deployment, and platform setup in one integrated workflow—from local development to production.
Connect these docs to Claude, V

In [7]:
# Similarity search
results = vectorstore.similarity_search_with_score(query,k=3)
for doc,score in results:
    print('score:',score)
    print(doc.page_content[:500])
    print("="*100)

score: 1.1712986
​More ways to build
FleetDesign and deploy AI agents visually without writing code.Build an agentPrompt engineeringIterate on prompts with built-in versioning and collaboration to ship improvements faster.Test your promptsLangSmith CLIQuery and manage traces, datasets, experiments, and more from the terminal.Use the CLIStudioUse a visual interface to design, test, and refine applications end-to-end.Develop with Studio
​Infrastructure
Platform setupUse LangSmith in managed cloud, in a self-hosted
score: 1.2073777
Connect these docs to Claude, VSCode, and more via MCP for real-time answers.Edit this page on GitHub or file an issue.Was this page helpful?YesNoPreviousAgent lifecycleNext⌘IDocs by LangChain home pagegithubxlinkedinyoutubeResourcesForumChangelogLangChain AcademyContact SalesCompanyHomeTrust CenterCareersBloggithubxlinkedinyoutube
score: 1.2086375
LangSmith overview - Docs by LangChainDocumentation IndexFetch the complete documentation index at: /llms.txtUse t

In [8]:
# Retriver
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k":3}
)
retriever_docs=retriever.invoke(query)
retriever_docs

[Document(id='9cc0479b-5f22-400d-bcaf-a68bb96ce793', metadata={'source': 'https://docs.smith.langchain.com/tutorials/Administrators/manage_spend', 'title': 'LangSmith overview - Docs by LangChain', 'language': 'en'}, page_content='\u200bMore ways to build\nFleetDesign and deploy AI agents visually without writing code.Build an agentPrompt engineeringIterate on prompts with built-in versioning and collaboration to ship improvements faster.Test your promptsLangSmith CLIQuery and manage traces, datasets, experiments, and more from the terminal.Use the CLIStudioUse a visual interface to design, test, and refine applications end-to-end.Develop with Studio\n\u200bInfrastructure\nPlatform setupUse LangSmith in managed cloud, in a self-hosted environment, or hybrid to match your infrastructure and compliance needs.Choose how to set up LangSmithSecurity & complianceLangSmith meets the highest standards of data security and privacy with HIPAA, SOC 2 Type 2, and GDPR compliance. Meet with our tea

In [9]:
# send context to LLM
from langchain_ollama import OllamaLLM
llm=OllamaLLM(model="gemma:2b")
context="\n\n".join([doc.page_content for doc in retriever_docs])
prompt=f"Answer the following question based on the context below:\n\nContext: {context}\n\nQuestion: {query}\n\nAnswer:"
response=llm.invoke(prompt)
print(response)

The context does not provide any information about spending controls, so I cannot answer this question from the provided context.


In [13]:
# pip install Langchain_community
from langchain_community.document_loaders import WebBaseLoader

loader = WebBaseLoader(
    "https://docs.smith.langchain.com/tutorials/Administrators/manage_spend"
)

docs = loader.load()

print(docs[0].page_content[:1000])
from langchain_ollama import OllamaEmbeddings

embeddings = OllamaEmbeddings(
    model="nomic-embed-text"
)
from langchain_community.vectorstores import FAISS
vectorstore=FAISS.from_documents(docs, embeddings)
print("FAISS vectorstore created")
query="how do I create an API key..?"
results=vectorstore.similarity_search(query,k=3)
for doc in results:
    print(doc.page_content)
    print("="*100)
results=vectorstore.similarity_search_with_score(query,k=3)
for doc,scores in results:
    print("score:",score)
    print(doc.page_content,scores)
    print("="*100)
results=vectorstore.similarity_search_with_score(query,k=3)
for doc,scores in results:
    print("score:",score)
    print(doc.page_content[:500])
    print("="*100)
retriever=vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={
        "k":3
    }
)
retrieved_docs=retriever.invoke(query)
print(retrieved_docs)
from langchain_ollama import OllamaLLM
llm=OllamaLLM(
    model="gemma:2b"
)
context="\n".join([doc.page_content for doc in retrieved_docs])
print(context)
prompt=f"""
Answer the question based on context below.
context: {context}
question: {query}
"""
response=llm.invoke(prompt)
print(response)


LangSmith overview - Docs by LangChainDocumentation IndexFetch the complete documentation index at: /llms.txtUse this file to discover all available pages before exploring further.Skip to main contentDocs by LangChain home pageHomeSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationLangSmith overviewHomeLangSmithAgent lifecycleLangSmith overviewCopy pageCopy pageLangSmith is a framework-agnostic platform for building, debugging, and deploying AI agents and LLM applications. Trace requests, evaluate outputs, test prompts, and manage deployments all in one place, with your agent stack.
​Get started
Create an accountSign up at smith.langchain.com (no credit card required).
You can log in with Google, GitHub, or email.Create an API keyGo to your Settings page → API Keys → Create API Key.
Copy the key and save it securely.Choose your integrationLangSmith works with many frameworks and providers. Browse available integrations to connect your stack.
Once your account and API k